# Fine-tune TrOCR (Printed → Handwritten) on Custom Dataset

This notebook fine-tunes the **TrOCR Sin Printed Text** model (from Hugging Face `danush99/Model_TrOCR-Sin-Printed-Text`) on handwritten text data, then saves checkpoints to `NoteBooks/checkPoints` and the final model to the Hub as **danush99/Model_TrOCR-Sin-Fined-Tuned-Handwritten-Text**.

- **Base model:** `danush99/Model_TrOCR-Sin-Printed-Text` from Hugging Face Hub (VisionEncoderDecoderModel, DeiT + SinBERT)
- **Dataset:** `NoteBooks/AllData` (handwritten-data train/test with `data.csv` + `images/`)
- **Checkpoints:** `NoteBooks/checkPoints`
- **Hub model:** `danush99/Model_TrOCR-Sin-Fined-Tuned-Handwritten-Text`

## 1. Paths and config

Run this once. Paths are relative to the **notebook directory** (assume notebook is run from `NoteBooks/`).

In [ ]:
import os
from pathlib import Path

# Assume notebook is in NoteBooks/; repo root is parent of NoteBooks
NOTEBOOK_DIR = Path(os.getcwd())
if (NOTEBOOK_DIR / "AllData").exists():
    pass  # running from NoteBooks
else:
    NOTEBOOK_DIR = NOTEBOOK_DIR / "NoteBooks" if (NOTEBOOK_DIR / "NoteBooks").exists() else NOTEBOOK_DIR

REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "AllData").exists() else NOTEBOOK_DIR

# Paths
DATA_ROOT = NOTEBOOK_DIR / "AllData" / "handwritten-data"
CHECKPOINT_DIR = NOTEBOOK_DIR / "checkPoints"
# BASE_MODEL_HUB_ID = "microsoft/trocr-base-handwritten"  # Load from Hugging Face Hub for printed text
BASE_MODEL_HUB_ID = "danush99/Model_TrOCR-Sin-Printed-Text"  # Load from Hugging Face Hub for handwritten text
# HUB_MODEL_ID = "danush99/Model_TrOCR-Sin-Printed-Text" # save model which trained for printed text
HUB_MODEL_ID = "danush99/Model_TrOCR-Sin-Fined-Tuned-Handwritten-Text" # save model which trained for handwritten text

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("BASE_MODEL_HUB_ID:", BASE_MODEL_HUB_ID)
print("HUB_MODEL_ID:", HUB_MODEL_ID)

DATA_ROOT: /Users/danusha/Documents/Uni-Work/MSC/FYP/Bsc/NoteBooks/AllData/handwritten-data
CHECKPOINT_DIR: /Users/danusha/Documents/Uni-Work/MSC/FYP/Bsc/NoteBooks/checkPoints
BASE_MODEL_HUB_ID: danush99/Model_TrOCR-Sin-Printed-Text
HUB_MODEL_ID: danush99/Model_TrOCR-Sin-Fined-Tuned-Handwritten-Text


## 2. GPU / device setup (Windows vs macOS)

**If you get ModuleNotFoundError: No module named 'torch'** → run the **Install dependencies** cell (section 3) first, then re-run this section.

- **Windows (NVIDIA):** uses `cuda` if available.
- **macOS (Apple Silicon M1/M2):** uses `mps` if available, else CPU.
- To force a device, run the second cell and uncomment the line for your OS.

In [20]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [21]:
import torch

# Auto-detect device: CUDA (Windows/Linux) or MPS (macOS Apple Silicon)
if torch.cuda.is_available():
    device = "cuda"
    print("Using GPU: CUDA")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
    print("Using GPU: MPS (macOS Apple Silicon)")
else:
    device = "cpu"
    print("Using CPU")

Using GPU: MPS (macOS Apple Silicon)


In [22]:
# Optional: force device (uncomment the line for your OS)
# device = "cuda"   # Windows / Linux with NVIDIA GPU
# device = "mps"    # macOS Apple Silicon
# device = "cpu"
print("device =", device)

device = mps


## 3. Install dependencies (run this first if torch is missing)

Run the cell below to install PyTorch and other packages. Do this **before** the GPU cell (section 2) if you see **ModuleNotFoundError: No module named 'torch'**.

In [23]:
!pip install torch transformers datasets evaluate pandas Pillow tf-keras

## 4. Imports

If you see **ValueError** about Keras 3 / tf_keras, run the **Install dependencies** cell (section 3) again (it includes `tf-keras`), then re-run this cell.

In [24]:
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    TrOCRProcessor,
    ViTImageProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)

## 5. Load processor and base model

Processor matches the printed model: SinBERT tokenizer + DeiT image processor. Model is loaded from Hugging Face Hub: `danush99/Model_TrOCR-Sin-Printed-Text`.

In [25]:
# Processor: same as printed model (SinBERT + DeiT)
tokenizer = AutoTokenizer.from_pretrained("NLPC-UOM/SinBERT-large")
feature_extractor = ViTImageProcessor.from_pretrained("facebook/deit-base-distilled-patch16-224")
processor = TrOCRProcessor(image_processor=feature_extractor, tokenizer=tokenizer)

# Base model from Hugging Face Hub
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL_HUB_ID)

# Set special tokens and generation config (required for decoding)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

model.to(device)
print("Processor and model loaded. Model from:", BASE_MODEL_HUB_ID)

Processor and model loaded. Model from: danush99/Model_TrOCR-Sin-Printed-Text


## 6. Load dataset (AllData)

Train/test CSVs: `file_name` and `text`. Images live in `images/` as `{file_name}.png`.

In [26]:
train_df = pd.read_csv(DATA_ROOT / "train" / "data.csv")
test_df = pd.read_csv(DATA_ROOT / "test" / "data.csv")
print("Train samples:", len(train_df))
print("Test samples:", len(test_df))
print(train_df.head())

Train samples: 908
Test samples: 227
    file_name                         text
0     image_0                        පුරුෂ
1    image_10                 නීල් රූපසිංහ
2   image_100                      සහෝදරයා
3  image_1002  පෙරේරලාගේ සමන් කුමාර පීරිස්
4  image_1003       පී . සමන් කුමාර පීරිස්


## 7. Dataset class

PyTorch Dataset that loads image from `root_dir/{file_name}.png` and labels from the processor tokenizer.

In [27]:
class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=256):
        self.root_dir = Path(root_dir)
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df["file_name"].iloc[idx]
        text = self.df["text"].iloc[idx]
        image_path = self.root_dir / f"{file_name}.png"
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
        ).input_ids
        labels = [
            label if label != self.processor.tokenizer.pad_token_id else -100
            for label in labels
        ]
        return {"pixel_values": pixel_values.squeeze(0), "labels": torch.tensor(labels)}

In [28]:
train_dataset = IAMDataset(
    root_dir=DATA_ROOT / "train" / "images",
    df=train_df,
    processor=processor,
)
eval_dataset = IAMDataset(
    root_dir=DATA_ROOT / "test" / "images",
    df=test_df,
    processor=processor,
)
print("Train size:", len(train_dataset), "| Eval size:", len(eval_dataset))

Train size: 908 | Eval size: 227


## 8. Training arguments

Checkpoints are saved under `NoteBooks/checkPoints`. On macOS (MPS), set `fp16=False` if you see errors.

In [34]:
pip install --upgrade accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 2.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [35]:
pip install --upgrade transformers[torch]

zsh:1: no matches found: transformers[torch]
Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install "transformers[torch]"

Note: you may need to restart the kernel to use updated packages.


In [41]:
!{sys.executable} -m pip install --upgrade accelerate transformers[torch]

zsh:1: no matches found: transformers[torch]


In [ ]:
import accelerate
import transformers
import sys
print(accelerate.__version__)
print(transformers.__version__)
print(sys.executable)

# Use fp16 only with CUDA; MPS does not support fp16
use_fp16 = device == "cuda"

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy="steps",          # how often to evaluate ("no", "steps", "epoch")
    save_strategy="steps",          # how often to save checkpoints
    logging_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    predict_with_generate=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=use_fp16,
    load_best_model_at_end=True,
    metric_for_best_model="eval_cer",
    greater_is_better=False,
    num_train_epochs=100,
    push_to_hub=False,
)

1.12.0
4.57.3
/opt/anaconda3/bin/python


TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## 9. Compute metrics (CER)

Character Error Rate for evaluation.

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    label_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

## 10. Trainer and training

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.image_processor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate(eval_dataset)
print(eval_results)

## 11. Save model and push to Hub

Saves the fine-tuned model locally, then pushes to **danush99/Model_TrOCR-Sin-Fined-Tuned-Handwritten-Text**. Log in to Hugging Face first if needed: `huggingface-cli login`.

In [ ]:
# Save locally (e.g. next to checkpoints)
final_model_dir = CHECKPOINT_DIR / "final_model"
trainer.save_model(str(final_model_dir))
processor.save_pretrained(str(final_model_dir))
print("Saved to", final_model_dir)

In [ ]:
# Push to Hugging Face Hub (requires login: huggingface-cli login)
trainer.push_to_hub(HUB_MODEL_ID)

## 12. Inference (optional)

Load a test image and run the model.

In [ ]:
# Example: run on first test image
test_image_path = DATA_ROOT / "test" / "images" / (test_df["file_name"].iloc[0] + ".png")
image = Image.open(test_image_path).convert("RGB")
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
generated_ids = model.generate(pixel_values)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Predicted:", generated_text)
print("Ground truth:", test_df["text"].iloc[0])